# 02 · FastText Implementation

In [1]:
import sys
sys.path.append('..')

import os
import re
import gzip
import shutil

import numpy as np
import pandas as pd
import torch
import urllib.request

from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# https://radimrehurek.com/gensim/models/fasttext.html
from gensim.models.fasttext import load_facebook_model

print(f'device : {"cuda" if torch.cuda.is_available() else "cpu"}')

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/dl/lib64/python3.9/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/dl/lib64/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/dl/lib64/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/dl/lib64/python3.9/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should en

/home/hice1/rdurai6/scratch/project/graph-semantic-research-main/dl/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device : cuda


In [2]:
input_dir  = '../data/'
output_dir = os.path.expanduser('../outputs/')
model_dir  = os.path.expanduser('../models/')
os.makedirs(output_dir, exist_ok=True)
os.makedirs(model_dir,  exist_ok=True)

FASTTEXT_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz'
FASTTEXT_BIN = os.path.join(model_dir, 'cc.en.300.bin')

save_path = os.path.join(output_dir, 'title_embeddings_fasttext.npy')

In [3]:
if os.path.exists(FASTTEXT_BIN):
    print(f'Model already present: {FASTTEXT_BIN}')
else:
    gz_path = FASTTEXT_BIN + '.gz'
    if not os.path.exists(gz_path):
        print(f'Downloading {FASTTEXT_URL}')
        urllib.request.urlretrieve(FASTTEXT_URL, gz_path)
    print('Decompressing')
    with gzip.open(gz_path, 'rb') as f_in, open(FASTTEXT_BIN, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(gz_path)

print('Model downloaded successfully!')

Model already present: ../models/cc.en.300.bin
Model downloaded successfully!


In [4]:
df = pd.read_parquet(input_dir + 'amazon_products.parquet')
print('Data loaded successfully!')

Data loaded successfully!


In [6]:
from scripts.embedding_pipeline_st import EmbeddingPipeline
from typing import Optional

REGEX = re.compile(r'[^a-z0-9\s]')

def tokenise(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return REGEX.sub(' ', text.lower()).split()


class FastTextEmbeddingPipeline(EmbeddingPipeline):
    """
    Subclass of EmbeddingPipeline that replaces the SentenceTransformer backend with a gensim fastText model.

    model_name: path to a Facebook fastText .bin file
    l2_normalise: L2-normalise each embedding (default True)
    """

    def __init__(self, model_name: str, l2_normalise: bool = True):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model_name = model_name
        self.l2_normalise = l2_normalise
        self.model = load_facebook_model(model_name)
        self._dim  = self.model.vector_size
        print(f'Model loaded')
        print(f'vocab={len(self.model.wv.key_to_index):,}')
        print(f'dim={self._dim}')

    def embed_sentences(self, sentences: list, batch_size: int) -> np.ndarray:
        N = len(sentences)
        all_embeds = np.zeros((N, self._dim), dtype=np.float32)

        for start in tqdm(range(0, N, batch_size), unit='batch'):
            end = min(start + batch_size, N)
            for i, text in enumerate(sentences[start:end]):
                tokens = tokenise(text)
                if not tokens:
                    continue

                vectors = np.stack([self.model.wv[tok] for tok in tokens])
                all_embeds[start+i] = vectors.mean(axis=0)

        if self.l2_normalise:
            t = torch.from_numpy(all_embeds)
            all_embeds = torch.nn.functional.normalize(t, p=2, dim=1).numpy()

        return all_embeds

    def compute_embeddings(self, df: pd.DataFrame, column: str, batch_size: int = 128, save_path: Optional[str] = None) -> np.ndarray:
        if column not in df.columns:
            raise ValueError(f"Column '{column}' not found in DataFrame.")

        sentences = df[column].fillna('').astype(str).tolist()

        print(f'Starting embedding process for {len(sentences)} items')
        embeddings = self.embed_sentences(sentences, batch_size)

        print(f'Computation complete. Embedding shape: {embeddings.shape}')

        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            np.save(save_path, embeddings)
            print(f'Embeddings saved to: {save_path}')

        return embeddings

In [7]:
pipeline = FastTextEmbeddingPipeline(model_name=FASTTEXT_BIN)
embeddings = pipeline.compute_embeddings(df=df, column='title', batch_size=128, save_path=save_path)

Model loaded
vocab=2,000,000
dim=300
Starting embedding process for 548552 items


100%|██████████| 4286/4286 [00:23<00:00, 185.70batch/s]


Computation complete. Embedding shape: (548552, 300)
Embeddings saved to: ../outputs/title_embeddings_fasttext.npy


In [8]:
norms = np.linalg.norm(embeddings, axis=1)

print(f'Embedding dim : {embeddings.shape[1]}')
print(f'Total nodes : {embeddings.shape[0]:,}')
print(f'Zero-vector rows (titles with no recognisable tokens) : {(norms < 1e-8).sum():,}')
print(f'Norm mean / std : {norms.mean():.4f} / {norms.std():.4f}')

Embedding dim : 300
Total nodes : 548,552
Zero-vector rows (titles with no recognisable tokens) : 5,875
Norm mean / std : 0.9893 / 0.1029


In [9]:
def test_embeddings(df, embeddings, query_index, top_k=5):
    query_title = df.iloc[query_index]['title']
    query_vec = embeddings[query_index].reshape(1, -1)

    print(f'--- QUERY PRODUCT (Index {query_index}) ---')
    print(f'Title: {query_title}\n')

    scores  = cosine_similarity(query_vec, embeddings).flatten()
    indices = np.argsort(scores)[-(top_k + 1):][::-1]

    print(f'--- TOP {top_k} SIMILAR PRODUCTS ---')
    for i in indices:
        if i == query_index:
            continue

        similarity  = scores[i]
        match_title = df.iloc[i]['title']
        print(f'[{similarity:.4f}] {match_title}')

In [10]:
test_index = 100
test_embeddings(df, embeddings, query_index=test_index)

--- QUERY PRODUCT (Index 100) ---
Title: Guide to Effective Staff Development in Health Care Organizations : A Systems Approach to Successful Training (J-B AHA Press)

--- TOP 5 SIMILAR PRODUCTS ---
[0.9425] Disease Management : A Systems Approach to Improving Patient Outcomes (J-B AHA Press)
[0.9349] Evolving Practices in Human Resource Management : Responses to a Changing World of Work (J-B SIOP Professional Practice Series)
[0.9234] Insights in Decision Making : A Tribute to Hillel J. Einhorn
[0.9162] Spelling Smart! : A Ready-to-Use Activities Program for Students with Spelling Difficulties (J-B Ed: Ready-to-Use Activities)
[0.9147] Smart Alliances : A Practical Guide to Repeatable Success (J-B BAH Strategy & Business Series)
